## 카드 데이터 형식, 컬럼 확인

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path("/content/drive/MyDrive/bigcontest-2026")
CARD_DIR = BASE / "data/raw/card"
print(list(CARD_DIR.iterdir()))

In [ ]:
for f in sorted(CARD_DIR.glob("*.txt")):
    raw = f.read_bytes()[:1000]
    for enc in ["utf-8-sig", "cp949"]:
        try:
            text = raw.decode(enc)
            break
        except UnicodeDecodeError:
            continue
    print(f"=== {f.name} ===")
    print("파일 크기:", round(f.stat().st_size / 1024**2, 1), "MB")
    print("인코딩:", enc)
    print(repr(text[:300]))
    print()

In [ ]:
import pandas as pd

def load_card(name):
    df = pd.read_csv(CARD_DIR / name, sep="\t", encoding="cp949", dtype=str)
    df["TS_AT"] = pd.to_numeric(df["TS_AT"])
    df["USE_CNT"] = pd.to_numeric(df["USE_CNT"])
    return df

card1 = load_card("신한카드_빅콘테스트2026_데이터1.txt")
card2 = load_card("신한카드_빅콘테스트2026_데이터2.txt")

for name, df in [("데이터1", card1), ("데이터2", card2)]:
    print(f"\n========== {name} | {df.shape} ==========")
    print("기간:", df["TA_YMD"].min(), "~", df["TA_YMD"].max(),
          f"({df['TA_YMD'].nunique()}일)")
    print("완전 중복 행:", df.duplicated().sum())
    for c in df.columns:
        if c in ["TA_YMD", "TS_AT", "USE_CNT"]:
            continue
        vc = df[c].value_counts(dropna=False)
        print(f"\n[{c}] {len(vc)}개")
        print(vc.head(40).to_string())

In [ ]:
local = {"서울 강남구": "서울", "강원 춘천시": "강원"}

# 1. 마스킹 확인: 최소 결제건수
print("[최소 결제건수] 데이터1:", card1["USE_CNT"].min(), "| 데이터2:", card2["USE_CNT"].min())
print(card2["USE_CNT"].value_counts().sort_index().head(10).to_string())

# 2. 법인 매출 비중
corp = card1.groupby(card1["SEX_CCD"] == "법인")["TS_AT"].sum()
print(f"\n[법인 매출 비중] {corp[True] / corp.sum():.1%}")

# 3. 지역별 외지인 매출 비중
card2["visitor"] = card2["CLN_SGG_CD"] != card2["MCT_SGG_CD"].map(local)
v = card2.groupby(["MCT_SGG_CD", "visitor"])["TS_AT"].sum().unstack()
print("\n[외지인 매출 비중]")
print((v[True] / v.sum(axis=1)).map("{:.1%}".format).to_string())

# 4. 지역별 시간대 매출 비중
t = card1[card1["SEX_CCD"] != "법인"].pivot_table(
    index="MCT_SGG_CD", columns="TIME_GB", values="TS_AT", aggfunc="sum")
print("\n[시간대 매출 비중]")
print(t.div(t.sum(axis=1), axis=0).map("{:.1%}".format).to_string())

# 5. 두 파일 업종 차이
s1, s2 = set(card1["MCT_RY_CD"]), set(card2["MCT_RY_CD"])
print("\n[데이터1에만]", s1 - s2, "\n[데이터2에만]", s2 - s1)

# 6. 지역별 업종 매출 상위 25
top = card2.groupby(["MCT_SGG_CD", "MCT_RY_CD"])["TS_AT"].sum()
for r in local:
    share = top[r].sort_values(ascending=False) / top[r].sum()
    print(f"\n[{r} 업종 매출 상위 25]")
    print(share.head(25).map("{:.1%}".format).to_string())

In [ ]:
# 법인 제외
card1 = card1[card1["SEX_CCD"] != "법인"].copy()

# 업종별 지역 매출
ind = card2.pivot_table(index="MCT_RY_CD", columns="MCT_SGG_CD",
                        values="TS_AT", aggfunc="sum", fill_value=0)
ind = ind.div(ind.sum(axis=0), axis=1)  # 지역 내 매출 비중
ind.columns = ["춘천_비중", "강남_비중"]

# 초안 매핑 (확정 아님, 같이 수정)
draft = {
    "제외": ["세금공과금", "ZZ_나머지", "보험", "통신요금(이동,시내전화)",
            "통신요금(PC통신,무선호출)", "상품권/복권", "방문판매/다단계판매",
            "고속버스/철도/여객선", "예식장/결혼서비스"],
    "음식점": ["한식", "중식", "일식", "양식", "기타요식", "패스트푸드"],
    "카페·제과": ["커피전문점", "제과점"],
    "식료품소매": ["할인점/슈퍼마켓/양판점", "편의점", "기타식품", "정육점", "농수산물"],
    "패션·잡화": ["의복/의류", "패션잡화", "생활잡화/수입상품점", "화장품",
              "시계/귀금속", "스포츠/레저용품"],
    "대형유통": ["백화점", "쇼핑몰"],
    "의료": ["일반병원", "종합병원", "치과병원", "한의원", "약국", "기타의료"],
    "실내여가": ["영화/공연", "노래방", "게임방/오락실", "싸우나/목욕탕"],
    "스포츠": ["스포츠시설", "헬스장", "실내/실외골프장"],
    "숙박": ["모텔,여관,기타숙박"],
    "생활서비스": ["미용실", "세탁소"],
    "교육": ["학원/학습지", "서점"],
    "자동차": ["주유소", "LPG가스", "자동차서비스", "자동차용품", "주차장"],
}
m = {ind_name: g for g, names in draft.items() for ind_name in names}
ind["그룹(초안)"] = ind.index.map(m).fillna("미분류")

ind = ind.sort_values("강남_비중", ascending=False)
ind.to_csv(BASE / "docs/industry_mapping.csv", encoding="utf-8-sig")

print("미분류 업종:", (ind["그룹(초안)"] == "미분류").sum(), "개")
print(ind[ind["그룹(초안)"] == "미분류"].map(
    lambda x: f"{x:.2%}" if isinstance(x, float) else x).to_string())

In [ ]:
draft["스포츠"] = ["스포츠시설", "헬스장"]
draft["야외레저"] = ["종합레저타운/놀이동산", "실내/실외골프장"]
draft["숙박"] += ["호텔/콘도"]
draft["실내여가"] += ["유흥업소"]
draft["생활서비스"] += ["미용서비스", "안마/마사지", "동물병원", "애완동물"]
draft["식료품소매"] += ["주류판매"]
draft["패션·잡화"] += ["문화용품", "완구/아동용자전거", "화원", "악기/음반",
                    "수제용품점", "중고품판매점", "사무기기/문구용품"]
draft["대형유통"] += ["면세점", "기타유통", "전용매장", "체인점"]
draft["교육"] += ["독서실"]
draft["제외_내구재"] = ["컴퓨터/소프트웨어", "가전", "가구", "수입자동차",
                    "인테리어/건축자재/주방기구", "오토바이", "중고차판매"]
draft["제외"] += ["연구/번역서비스", "법률/사무서비스", "회계/변리서비스", "결제대행(PG)",
                "부동산중개", "학교등록금", "유치원", "장례식장/묘지/장의사",
                "보건소", "여행사/항공사", "택시"]

m = {name: g for g, names in draft.items() for name in names}
for df in (card1, card2):
    df["IND_GRP"] = df["MCT_RY_CD"].map(m).fillna("미분류")
print("미분류:", sorted(set(card1.loc[card1["IND_GRP"] == "미분류", "MCT_RY_CD"])))

# 분석 대상만 남기고 다시 확인
use2 = card2[~card2["IND_GRP"].str.startswith("제외")]

v = use2.groupby(["MCT_SGG_CD", "visitor"])["TS_AT"].sum().unstack()
print("\n[외지인 매출 비중 - 업종 정리 후]")
print((v[True] / v.sum(axis=1)).map("{:.1%}".format).to_string())

g = use2.pivot_table(index="IND_GRP", columns="MCT_SGG_CD", values="TS_AT", aggfunc="sum")
print("\n[그룹별 매출 비중]")
print(g.div(g.sum()).sort_values("강원 춘천시", ascending=False).map("{:.1%}".format).to_string())

vg = use2.groupby(["MCT_SGG_CD", "IND_GRP", "visitor"])["TS_AT"].sum().unstack()
print("\n[그룹별 외지인 비중]")
print((vg[True] / vg.sum(axis=1)).unstack(0).map("{:.0%}".format).to_string())

In [ ]:
OUT = BASE / "data/processed"

# 1) 분석 대상만 남긴 정제본 저장
c1 = card1[~card1["IND_GRP"].str.startswith("제외")].copy()
c2 = card2[~card2["IND_GRP"].str.startswith("제외")].copy()
for df in (c1, c2):
    df["date"] = pd.to_datetime(df["TA_YMD"], format="%Y%m%d")
    df["region"] = df["MCT_SGG_CD"].map({"서울 강남구": "강남", "강원 춘천시": "춘천"})
    df["AGE_CCD"] = df["AGE_CCD"].str.replace(" ", "")  # "20 대" → "20대"
c1.to_parquet(OUT / "card_time_clean.parquet", index=False)
c2.to_parquet(OUT / "card_visitor_clean.parquet", index=False)

# 2) 분석 단위 집계
time_panel = (c1.groupby(["date", "region", "IND_GRP", "TIME_GB"])
                [["TS_AT", "USE_CNT"]].sum().reset_index())
visitor_panel = (c2.groupby(["date", "region", "IND_GRP", "visitor"])
                   [["TS_AT", "USE_CNT"]].sum().reset_index())
time_panel.to_parquet(OUT / "panel_time.parquet", index=False)
visitor_panel.to_parquet(OUT / "panel_visitor.parquet", index=False)

# 3) 마스킹 영향 확인: 184일 중 데이터가 있는 날 수
cover = (c2.groupby(["region", "IND_GRP"])["date"].nunique()
           .unstack(0).sort_values("춘천"))
print("[업종 그룹별 데이터 존재 일수 (최대 184)]")
print(cover.to_string())

In [ ]:
# 외지인/지역민, 시간대로 쪼갰을 때 존재 일수
cv = visitor_panel.groupby(["region", "IND_GRP", "visitor"])["date"].nunique().unstack([0, 2])
ct = time_panel.groupby(["region", "IND_GRP", "TIME_GB"])["date"].nunique().unstack([0, 2])
print("[외지인 분리 시 존재 일수]\n", cv.to_string())
print("\n[시간대 분리 시 존재 일수]\n", ct.to_string())

# 일 매출 변동계수(표준편차/평균): 높을수록 들쭉날쭉
daily = visitor_panel.groupby(["region", "IND_GRP", "date"])["TS_AT"].sum()
cvar = daily.groupby(["region", "IND_GRP"]).agg(lambda s: s.std() / s.mean()).unstack(0)
print("\n[일 매출 변동계수]\n", cvar.round(2).to_string())